# 26b · GSE135779 · scRNA_seq (pseudobulk) · scoring these samples on the GSE65391 modules

Reads this study's `expression.rds` and the GSE65391 end product `module_loadings.csv` (for each module:
its genes and their weights; no GSE65391 expression values). Writes `projected_scores.rds` in
`data/run_artifacts/GSE135779/`.

**What is transferred.** The model fitted on the GSE65391 first visits (notebook 05b): for each module,
the genes and their weights (u₁ / d₁ of the module's first principal component). Scoring a new data set
with a model learned elsewhere is transfer learning by projection (Stein-O'Brien GL et al. Decomposing
cell identity for transfer learning across cellular measurements, platforms, tissues, and species.
*Cell Syst* 2019;8:395–411, doi:10.1016/j.cels.2019.04.004; Sharma G et al. projectR. *Bioinformatics*
2020;36:3592–3593, doi:10.1093/bioinformatics/btaa183).

**Scaling is local.** GSE65391 standardised each gene with its own first-visit mean and SD. These are
array intensities; here the values are RNA sequencing log counts on another scale, so the array's means
and SDs cannot be applied. Each gene is standardised within this study, with the mean and SD of its 33
children with SLE (the GSE65391 modules were fitted on children with SLE), and then weighted with the
array weights. The 11 healthy children are scored on the same scale:

  score = Σ over the module's genes measured here of z × weight

**Coverage.** Genes that this study does not measure are left out, and the share of each module's genes
used is reported.

**Check.** For each module, the projected score against the eigengene recomputed from this study's own
data, over the 33 children with SLE: Pearson r. A high r means the array's weighting and a local
refit agree on the module's axis in these samples.

In [1]:
source("../src/paths.R")
x  <- readRDS(art("GSE135779", "expression.rds"))
ld <- read.csv(art("GSE65391", "module_loadings.csv"))
E   <- x$E
sle <- x$meta$sle == 1
mu  <- rowMeans(E[, sle]); s <- apply(E[, sle], 1, sd)
Z   <- ((E - mu) / s)[s > 0, ]                           # genes x children, standardised on the SLE children
cov <- do.call(rbind, lapply(split(ld, ld$module), function(d)
  data.frame(module = d$module[1], genes_in_array_module = nrow(d), genes_here = sum(d$gene %in% rownames(Z)))))
cov$share_used <- round(cov$genes_here / cov$genes_in_array_module, 3)
cov[order(cov$share_used), ]

,module,genes_in_array_module,genes_here,share_used
,<chr>,<int>,<int>,<dbl>
black,black,250,90,0.360
yellow,yellow,416,202,0.486
green,green,301,156,0.518
salmon,salmon,55,32,0.582
lightcyan,lightcyan,36,22,0.611
magenta,magenta,103,77,0.748
greenyellow,greenyellow,84,63,0.750
brown,brown,426,342,0.803
red,red,268,217,0.810


**Result.** Coverage ranges from 36% (black) to 99% (purple); 11 of 16 modules have 75% or more.
Lightcyan (neutrophil granule) has 61%: several granule genes are too rarely counted in PBMC to pass the
filter of notebook 23.

In [2]:
scores <- sapply(split(ld, ld$module), function(d) {
  d <- d[d$gene %in% rownames(Z), ]
  colSums(Z[d$gene, , drop = FALSE] * d$weight)
})
dim(scores)

[1] 44 16

**Check.** Projected score against the locally recomputed eigengene of the same gene list.

In [3]:
suppressMessages(library(WGCNA))
ld_here <- ld[ld$gene %in% rownames(Z), ]
ME_local <- moduleEigengenes(t(E[ld_here$gene, sle]), colors = ld_here$module)$eigengenes
check <- data.frame(module = colnames(scores),
                    r_projected_vs_local = sapply(colnames(scores), function(mo) cor(scores[sle, mo], ME_local[, paste0("ME", mo)])),
                    row.names = NULL)
check$r_projected_vs_local <- round(check$r_projected_vs_local, 3)
check[order(check$r_projected_vs_local), ]

,module,r_projected_vs_local
,<chr>,<dbl>
5,green,0.773
15,turquoise,0.853
16,yellow,0.892
3,brown,0.945
8,magenta,0.965
13,salmon,0.974
2,blue,0.978
7,lightcyan,0.984
10,pink,0.985


**Result.** For 13 of 16 modules the projected score and the local refit agree at r ≥ 0.945, including
pink (interferon, 0.985), greenyellow (plasma cell, 0.999) and lightcyan (neutrophil granule, 0.984).
Three agree less: green (0.773), turquoise (0.853) and yellow (0.892).

In [4]:
saveRDS(list(scores = scores, coverage = cov, check = check), art("GSE135779", "projected_scores.rds"))